# 🚀 Train Multilingual Intent Classifier in Google Colab

Train XLM-RoBERTa model for Shona, Ndebele, and English customer service chatbot

**Dataset:** 16,088 records (7,929 Shona + 4,323 Ndebele + 3,836 English)  
**Model:** XLM-RoBERTa (multilingual)  
**Intents:** 23 customer service categories  

---

## 📋 Instructions

1. **Click Runtime → Change runtime type → GPU** (T4 recommended)
2. **Run cells in order** (they'll take ~5-10 minutes total)
3. **Download trained model** after completion
4. Copy model to your project's `backend/trained_model_enhanced_v5/` folder

---

## 1️⃣ Set Up Google Colab Environment

Check GPU/TPU availability and configure runtime

In [ ]:
# Check GPU availability
import torch
print("=" * 60)
print("GOOGLE COLAB ENVIRONMENT SETUP")
print("=" * 60)
print(f"\n✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU device: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠ No GPU detected. Go to Runtime → Change runtime type → GPU")
print("=" * 60)

## 2️⃣ Install Required Libraries and Dependencies

Install ML packages: PyTorch, Transformers, scikit-learn

In [ ]:
!pip install -q torch transformers scikit-learn numpy pandas requests -q
import os
os.chdir('/content')
print("✓ All packages installed successfully")

## 3️⃣ Mount Google Drive

Save and load files from Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully")

# Check available space
import shutil
total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"✓ Drive space available: {free / (1024**3):.2f} GB")

## 4️⃣ Load and Prepare Training Data

Load consolidated dataset and prepare train/validation splits

In [ ]:
import json
import random
from collections import defaultdict
from sklearn.model_selection import train_test_split

# OPTION 1: Load from URL (if you have the dataset on GitHub)
# Uncomment and modify the URL if needed:
# import urllib.request
# url = "https://raw.githubusercontent.com/YOUR_REPO/retraining_dataset_FINAL_CONSOLIDATED.json"
# urllib.request.urlretrieve(url, "dataset.json")

# OPTION 2: Upload dataset directly
# You can drag-and-drop the dataset file here, or use the upload below
from google.colab import files
print("📁 Please upload 'retraining_dataset_FINAL_CONSOLIDATED.json' file:")
uploaded = files.upload()

# Load dataset
dataset_file = list(uploaded.keys())[0] if uploaded else "retraining_dataset_FINAL_CONSOLIDATED.json"
with open(dataset_file, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(f"\n✓ Dataset loaded: {len(dataset)} records")

# Show statistics
lang_stats = defaultdict(int)
intent_stats = defaultdict(int)
for record in dataset:
    lang_stats[record.get('language', 'en')] += 1
    intent_stats[record.get('intent', 'unknown')] += 1

print(f"\nLanguage distribution:")
for lang, count in sorted(lang_stats.items()):
    print(f"  {lang}: {count} ({count/len(dataset)*100:.1f}%)")

print(f"\nTop 5 intents:")
for intent, count in sorted(intent_stats.items(), key=lambda x: -x[1])[:5]:
    print(f"  {intent}: {count}")

In [ ]:
# Prepare data
texts = [record['text'] for record in dataset]
intents = [record['intent'] for record in dataset]

# Create label mapping
unique_intents = sorted(set(intents))
intent2id = {intent: idx for idx, intent in enumerate(unique_intents)}
id2intent = {idx: intent for idx, intent in enumerate(unique_intents)}

# Convert intents to IDs
labels = [intent2id[intent] for intent in intents]

print(f"✓ Number of intent classes: {len(unique_intents)}")
print(f"✓ Total samples: {len(texts)}")

# Split into train/validation (80/20)
texts_train, texts_val, labels_train, labels_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"\n✓ Train set: {len(texts_train)} samples")
print(f"✓ Validation set: {len(texts_val)} samples")

## 5️⃣ Build the Model

Initialize XLM-RoBERTa multilingual model for intent classification

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Model configuration
model_name = "xlm-roberta-base"
num_labels = len(unique_intents)

print(f"🤖 Loading model: {model_name}")
print(f"📊 Number of labels: {num_labels}")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2intent,
    label2id=intent2id
)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✓ Model loaded and moved to {device}")
print(f"✓ Tokenizer ready")

## 6️⃣ Configure Training Parameters

Set hyperparameters for model training

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score

# Custom Dataset class
class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

# Prepare datasets
train_dataset = IntentDataset(texts_train, labels_train, tokenizer)
val_dataset = IntentDataset(texts_val, labels_val, tokenizer)

print("✓ Datasets prepared")
print(f"  Train: {len(train_dataset)}")
print(f"  Validation: {len(val_dataset)}")

# Training parameters
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=2e-5,
    report_to="none"
)

print("\n📋 Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")

## 7️⃣ Train the Model

Execute training with GPU acceleration (~5-10 minutes)

In [ ]:
def compute_metrics(eval_pred):
    """Compute metrics for evaluation"""
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("🚀 Starting training...")
print("=" * 60)

# Train
train_result = trainer.train()

print("=" * 60)
print("✓ Training completed!")

## 8️⃣ Evaluate Model Performance

Assess model accuracy on validation data

In [ ]:
# Evaluate on validation set
eval_result = trainer.evaluate()

print("\n📊 EVALUATION RESULTS")
print("=" * 60)
print(f"Accuracy:  {eval_result['eval_accuracy']:.4f}")
print(f"Precision: {eval_result['eval_precision']:.4f}")
print(f"Recall:    {eval_result['eval_recall']:.4f}")
print(f"F1 Score:  {eval_result['eval_f1']:.4f}")
print("=" * 60)

# Test predictions on sample queries
print("\n🧪 Sample Predictions:")
print("-" * 60)

test_queries = [
    ("What is my balance?", "en"),
    ("Ndipei account balance yangu", "sn"),
    ("Ngifuna ukuthumela imali", "nd"),
    ("Can I transfer money?", "en"),
    ("Tumai mari kuAccount yangu", "sn"),
]

model.eval()
with torch.no_grad():
    for text, lang in test_queries:
        encoding = tokenizer(
            text,
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors="pt"
        )
        encoding = {k: v.to(device) for k, v in encoding.items()}
        outputs = model(**encoding)
        probabilities = torch.softmax(outputs.logits, dim=1)
        confidence, predicted_id = torch.max(probabilities, dim=1)
        predicted_intent = id2intent[predicted_id.item()]
        
        print(f"Text: '{text}' ({lang})")
        print(f"  → Intent: {predicted_intent}")
        print(f"  → Confidence: {confidence.item():.2%}")
        print()

## 9️⃣ Save and Export Results

Save trained model to Google Drive and prepare for download

In [ ]:
import os
import shutil
from pathlib import Path

# Save model to Colab storage
model_output_dir = './trained_model_enhanced_v5'
os.makedirs(model_output_dir, exist_ok=True)

print(f"💾 Saving model to {model_output_dir}...")
model.save_pretrained(model_output_dir)
tokenizer.save_pretrained(model_output_dir)

# Save label mappings
label_mappings = {
    'id2label': id2intent,
    'label2id': intent2id
}
with open(f'{model_output_dir}/label_mappings.json', 'w') as f:
    json.dump(label_mappings, f, indent=2)

print("✓ Model saved locally")

# Option 1: Save to Google Drive
print("\n📤 Copying to Google Drive...")
drive_model_dir = '/content/drive/MyDrive/trained_model_enhanced_v5'
if os.path.exists(drive_model_dir):
    shutil.rmtree(drive_model_dir)
shutil.copytree(model_output_dir, drive_model_dir)
print(f"✓ Model saved to Drive: {drive_model_dir}")

# Option 2: Create zip for download
print("\n📦 Creating zip file for download...")
shutil.make_archive('trained_model_enhanced_v5', 'zip', model_output_dir)
print("✓ Zip file created: trained_model_enhanced_v5.zip")

# List important files
print("\n📂 Model Contents:")
for root, dirs, files in os.walk(model_output_dir):
    level = root.replace(model_output_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Show first 5 files
        size = os.path.getsize(os.path.join(root, file)) / 1024
        print(f'{subindent}{file} ({size:.1f}KB)')
    if len(files) > 5:
        print(f'{subindent}... and {len(files)-5} more files')

## ✅ Next Steps

After training completes:

### 1. Download the Model
- Download `trained_model_enhanced_v5.zip` from Colab
- Extract to your project: `backend/trained_model_enhanced_v5/`

### 2. Update Your Project
```bash
# Copy model files to your project
cp -r trained_model_enhanced_v5 backend/trained_model_enhanced_v5
```

### 3. Test the Model
```bash
# Run evaluation
python backend/scripts/evaluate_intent_model.py \
  --model-path backend/trained_model_enhanced_v5 \
  --dataset backend/generated/retraining_dataset_FINAL_CONSOLIDATED.json
```

### 4. Deploy to Production
- Restart your backend server
- The chatbot will automatically use the new trained model
- Monitor response accuracy improvements for Shona & Ndebele

---

## 📊 Training Summary

**Dataset:** 16,088 records
- Shona: 7,929 (49.3%)
- Ndebele: 4,323 (26.9%)
- English: 3,836 (23.8%)

**Model:** XLM-RoBERTa Base
**Intents:** 23 categories
**Training Time:** ~5-10 minutes (GPU)
**Expected Improvement:** +10-20% accuracy for Shona/Ndebele

---

🎉 **Training completed! Your chatbot is now ready with improved multilingual support.**